In [1]:
# pip install requests pandas python-dateutil
import requests, math, time
import pandas as pd
from datetime import datetime

FMP_BASE = "https://financialmodelingprep.com/api/v3"

def _get_json(url, params=None, retries=2, sleep=0.7):
    for i in range(retries+1):
        try:
            r = requests.get(url, params=params, timeout=20)
            if r.status_code == 200:
                return r.json()
        except Exception:
            pass
        time.sleep(sleep)
    return None

def get_income_q(ticker, api_key, limit=1):
    """최근 분기 손익계산서(Quarterly): revenue, operatingIncome"""
    url = f"{FMP_BASE}/income-statement/{ticker}"
    js = _get_json(url, {"period":"quarter", "limit":limit, "apikey":api_key})
    if not js:
        return None
    # 첫 행만 사용(가장 최근 분기)
    row = js[0]
    return {
        "revenue": row.get("revenue"),
        "operatingIncome": row.get("operatingIncome"),
        "date": row.get("date")
    }

def get_roa_ttm(ticker, api_key):
    """ROA 우선 TTM에서, 없으면 최신 key-metrics에서 보완"""
    # 1) TTM
    url = f"{FMP_BASE}/key-metrics-ttm/{ticker}"
    js = _get_json(url, {"apikey": api_key})
    if js and isinstance(js, list) and len(js) > 0:
        roa = js[0].get("roaTTM")
        if roa is not None:
            return roa
    # 2) 일반 key-metrics (분기/연간 혼재 → 가장 최신 값 사용)
    url = f"{FMP_BASE}/key-metrics/{ticker}"
    js = _get_json(url, {"period":"quarter", "limit":1, "apikey": api_key})
    if js and isinstance(js, list) and len(js) > 0:
        roa = js[0].get("roa")
        if roa is not None:
            return roa
    return None

def get_market_cap(ticker, api_key):
    """시가총액: quote에서 marketCap 사용"""
    url = f"{FMP_BASE}/quote/{ticker}"
    js = _get_json(url, {"apikey": api_key})
    if js and isinstance(js, list) and len(js) > 0:
        return js[0].get("marketCap")
    return None

def format_billions(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    try:
        return round(x / 1e9, 2)
    except Exception:
        return x

def collect_snapshot(tickers, api_key):
    records = []
    for tk in tickers:
        inc = get_income_q(tk, api_key, limit=1)
        roa = get_roa_ttm(tk, api_key)
        mcap = get_market_cap(tk, api_key)

        if inc:
            rev = inc["revenue"]
            oi  = inc["operatingIncome"]
            opm = (oi / rev * 100.0) if (oi is not None and rev and rev != 0) else None
            date = inc["date"]
        else:
            rev = oi = opm = date = None

        records.append({
            "Ticker": tk,
            "Quarter(Date)": date,
            "Revenue (USD Bn)": format_billions(rev),
            "Operating Income (USD Bn)": format_billions(oi),
            "Operating Margin (%)": None if opm is None else round(opm, 1),
            "ROA (TTM, %)": None if roa is None else round(roa*100 if roa < 1 else roa, 2),
            "Market Cap (USD Bn)": format_billions(mcap)
        })
    df = pd.DataFrame(records)
    # 보기 좋게 정렬
    cols = ["Ticker","Quarter(Date)","Revenue (USD Bn)","Operating Income (USD Bn)",
            "Operating Margin (%)","ROA (TTM, %)","Market Cap (USD Bn)"]
    return df[cols]

# ===== 사용 예시 =====
# API 키 입력
API_KEY = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"

# 포트폴리오 예시 (질문에 맞춤)
tickers = ["AVGO", "GOOG", "APH"]

df = collect_snapshot(tickers, API_KEY)
print(df.to_string(index=False))

# 필요 시 CSV로 저장
# df.to_csv("fmp_quarter_snapshot.csv", index=False, encoding="utf-8")


Ticker Quarter(Date)  Revenue (USD Bn)  Operating Income (USD Bn)  Operating Margin (%) ROA (TTM, %)  Market Cap (USD Bn)
  AVGO    2025-11-02             18.02                       7.51                  41.7         None              1625.12
  GOOG    2025-12-31            113.90                      36.00                  31.6         None              3765.07
   APH    2025-12-31              6.44                       1.77                  27.5         None               176.31
